# Read Bronze data

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

bronze_table = "learn_get_started_de_palvikrant235.taxi_etl_project.bronze_taxi_incremental"

silver_table = "learn_get_started_de_palvikrant235.taxi_etl_project.silver_taxi_incremental"

df_bronze = spark.table(bronze_table)

display(df_bronze.limit(10))

id,vendor_id,pickup_datetime,dropoff_datetime,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,store_and_fwd_flag,trip_duration
id1589506,2,2016-04-02T17:41:37.000Z,2016-04-02T17:57:33.000Z,5,-73.98480987548828,40.77939987182617,-73.9590072631836,40.819759368896484,N,956
id1081378,1,2016-05-17T03:41:22.000Z,2016-05-17T03:52:33.000Z,1,-74.00285339355469,40.7249755859375,-73.98432922363281,40.6916618347168,N,671
id2988321,2,2016-04-01T08:19:39.000Z,2016-04-01T08:27:36.000Z,1,-73.97863006591797,40.74494171142578,-73.98856353759766,40.73725891113281,N,477
id2759902,1,2016-06-27T06:00:56.000Z,2016-06-27T06:43:14.000Z,1,-73.87870025634766,40.73727035522461,-73.93812561035156,40.82817840576172,N,2538
id0965960,1,2016-03-22T08:02:22.000Z,2016-03-22T08:48:16.000Z,1,-73.87107849121094,40.773677825927734,-73.9829330444336,40.735069274902344,N,2754
id2572605,2,2016-01-12T23:59:57.000Z,2016-01-13T00:08:19.000Z,1,-74.00679016113281,40.740631103515625,-73.98470306396484,40.76845932006836,N,502
id1318259,2,2016-02-13T21:36:39.000Z,2016-02-13T21:50:20.000Z,1,-73.99723052978516,40.72539138793945,-73.98433685302734,40.717018127441406,N,821
id0687270,1,2016-01-08T12:24:26.000Z,2016-01-08T12:54:03.000Z,1,-73.97821807861328,40.76658630371094,-73.92739868164062,40.76282501220703,N,1777
id1749451,2,2016-06-14T08:02:56.000Z,2016-06-14T08:17:43.000Z,6,-73.98127746582031,40.74740982055664,-73.98404693603516,40.7592887878418,N,887
id2940014,2,2016-01-01T13:21:35.000Z,2016-01-01T13:39:17.000Z,1,-73.9476318359375,40.775150299072266,-73.9970703125,40.733951568603516,N,1062


In [0]:
null_counts = df_bronze.select([
    F.count(
        F.when(
            F.col(c).isNull() |
            (F.trim(F.col(c).cast("string")) == ""),
            c
        )
    ).alias(c)
    for c in df_bronze.columns
])

display(null_counts)

id,vendor_id,pickup_datetime,dropoff_datetime,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,store_and_fwd_flag,trip_duration
0,0,0,0,0,0,0,0,0,0,0


In [0]:
display(
    df_bronze.select(
        "trip_duration",
        "passenger_count",
        "pickup_longitude",
        "pickup_latitude",
        "dropoff_longitude",
        "dropoff_latitude"
    ).describe()
)

summary,trip_duration,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude
count,1458644,1458644,1458644,1458644,1458644,1458644
mean,959.4922729603659,1.6645295219395548,-73.97348630489282,40.750920908391734,-73.9734159469458,40.7517995149002
stddev,5237.431724497604,1.3142421678231166,0.07090185842270319,0.03288118625763302,0.07064326809720138,0.03589055560563706
min,1,0,-121.93334197998047,34.35969543457031,-121.93330383300781,32.1811408996582
max,3526282,9,-61.33552932739258,51.88108444213867,-61.33552932739258,43.92102813720703


In [0]:
df_silver = df_bronze

In [0]:
print("Total Bronze Records:", df_bronze.count())

Total Bronze Records: 1458644


# Adding New Column - "Price"

In [0]:
from pyspark.sql import functions as F

df_silver = df_silver.withColumn(
    "Price",
    F.round(F.col("trip_duration") / 60 * 2 , 2)
)
display(df_silver)

id,vendor_id,pickup_datetime,dropoff_datetime,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,store_and_fwd_flag,trip_duration,Price
id1589506,2,2016-04-02T17:41:37.000Z,2016-04-02T17:57:33.000Z,5,-73.98480987548828,40.77939987182617,-73.9590072631836,40.819759368896484,N,956,31.87
id1081378,1,2016-05-17T03:41:22.000Z,2016-05-17T03:52:33.000Z,1,-74.00285339355469,40.7249755859375,-73.98432922363281,40.6916618347168,N,671,22.37
id2988321,2,2016-04-01T08:19:39.000Z,2016-04-01T08:27:36.000Z,1,-73.97863006591797,40.74494171142578,-73.98856353759766,40.73725891113281,N,477,15.9
id2759902,1,2016-06-27T06:00:56.000Z,2016-06-27T06:43:14.000Z,1,-73.87870025634766,40.73727035522461,-73.93812561035156,40.82817840576172,N,2538,84.6
id0965960,1,2016-03-22T08:02:22.000Z,2016-03-22T08:48:16.000Z,1,-73.87107849121094,40.773677825927734,-73.9829330444336,40.735069274902344,N,2754,91.8
id2572605,2,2016-01-12T23:59:57.000Z,2016-01-13T00:08:19.000Z,1,-74.00679016113281,40.740631103515625,-73.98470306396484,40.76845932006836,N,502,16.73
id1318259,2,2016-02-13T21:36:39.000Z,2016-02-13T21:50:20.000Z,1,-73.99723052978516,40.72539138793945,-73.98433685302734,40.717018127441406,N,821,27.37
id0687270,1,2016-01-08T12:24:26.000Z,2016-01-08T12:54:03.000Z,1,-73.97821807861328,40.76658630371094,-73.92739868164062,40.76282501220703,N,1777,59.23
id1749451,2,2016-06-14T08:02:56.000Z,2016-06-14T08:17:43.000Z,6,-73.98127746582031,40.74740982055664,-73.98404693603516,40.7592887878418,N,887,29.57
id2940014,2,2016-01-01T13:21:35.000Z,2016-01-01T13:39:17.000Z,1,-73.9476318359375,40.775150299072266,-73.9970703125,40.733951568603516,N,1062,35.4


In [0]:
%sql
select * from learn_get_started_de_palvikrant235.taxi_etl_project.bronze_taxi_trips

id,vendor_id,pickup_datetime,dropoff_datetime,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,store_and_fwd_flag,trip_duration
id1589506,2,2016-04-02T17:41:37.000Z,2016-04-02T17:57:33.000Z,5,-73.98480987548828,40.77939987182617,-73.9590072631836,40.819759368896484,N,956
id1081378,1,2016-05-17T03:41:22.000Z,2016-05-17T03:52:33.000Z,1,-74.00285339355469,40.7249755859375,-73.98432922363281,40.6916618347168,N,671
id2988321,2,2016-04-01T08:19:39.000Z,2016-04-01T08:27:36.000Z,1,-73.97863006591797,40.74494171142578,-73.98856353759766,40.73725891113281,N,477
id2759902,1,2016-06-27T06:00:56.000Z,2016-06-27T06:43:14.000Z,1,-73.87870025634766,40.73727035522461,-73.93812561035156,40.82817840576172,N,2538
id0965960,1,2016-03-22T08:02:22.000Z,2016-03-22T08:48:16.000Z,1,-73.87107849121094,40.773677825927734,-73.9829330444336,40.735069274902344,N,2754
id2572605,2,2016-01-12T23:59:57.000Z,2016-01-13T00:08:19.000Z,1,-74.00679016113281,40.740631103515625,-73.98470306396484,40.76845932006836,N,502
id1318259,2,2016-02-13T21:36:39.000Z,2016-02-13T21:50:20.000Z,1,-73.99723052978516,40.72539138793945,-73.98433685302734,40.717018127441406,N,821
id0687270,1,2016-01-08T12:24:26.000Z,2016-01-08T12:54:03.000Z,1,-73.97821807861328,40.76658630371094,-73.92739868164062,40.76282501220703,N,1777
id1749451,2,2016-06-14T08:02:56.000Z,2016-06-14T08:17:43.000Z,6,-73.98127746582031,40.74740982055664,-73.98404693603516,40.7592887878418,N,887
id2940014,2,2016-01-01T13:21:35.000Z,2016-01-01T13:39:17.000Z,1,-73.9476318359375,40.775150299072266,-73.9970703125,40.733951568603516,N,1062


In [0]:
from delta.tables import DeltaTable

silver_table = "learn_get_started_de_palvikrant235.taxi_etl_project.silver_taxi_incremental"

if spark.catalog.tableExists(silver_table):

    delta_target = DeltaTable.forName(spark, silver_table)

    (
        delta_target.alias("target")
        .merge(
            df_silver.alias("source"),
            "target.id = source.id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    print("Silver MERGE completed successfully")

else:
    (
        df_silver.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(silver_table)
    )

display(df_silver)

id,vendor_id,pickup_datetime,dropoff_datetime,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,store_and_fwd_flag,trip_duration,Price
id1589506,2,2016-04-02T17:41:37.000Z,2016-04-02T17:57:33.000Z,5,-73.98480987548828,40.77939987182617,-73.9590072631836,40.819759368896484,N,956,31.87
id1081378,1,2016-05-17T03:41:22.000Z,2016-05-17T03:52:33.000Z,1,-74.00285339355469,40.7249755859375,-73.98432922363281,40.6916618347168,N,671,22.37
id2988321,2,2016-04-01T08:19:39.000Z,2016-04-01T08:27:36.000Z,1,-73.97863006591797,40.74494171142578,-73.98856353759766,40.73725891113281,N,477,15.9
id2759902,1,2016-06-27T06:00:56.000Z,2016-06-27T06:43:14.000Z,1,-73.87870025634766,40.73727035522461,-73.93812561035156,40.82817840576172,N,2538,84.6
id0965960,1,2016-03-22T08:02:22.000Z,2016-03-22T08:48:16.000Z,1,-73.87107849121094,40.773677825927734,-73.9829330444336,40.735069274902344,N,2754,91.8
id2572605,2,2016-01-12T23:59:57.000Z,2016-01-13T00:08:19.000Z,1,-74.00679016113281,40.740631103515625,-73.98470306396484,40.76845932006836,N,502,16.73
id1318259,2,2016-02-13T21:36:39.000Z,2016-02-13T21:50:20.000Z,1,-73.99723052978516,40.72539138793945,-73.98433685302734,40.717018127441406,N,821,27.37
id0687270,1,2016-01-08T12:24:26.000Z,2016-01-08T12:54:03.000Z,1,-73.97821807861328,40.76658630371094,-73.92739868164062,40.76282501220703,N,1777,59.23
id1749451,2,2016-06-14T08:02:56.000Z,2016-06-14T08:17:43.000Z,6,-73.98127746582031,40.74740982055664,-73.98404693603516,40.7592887878418,N,887,29.57
id2940014,2,2016-01-01T13:21:35.000Z,2016-01-01T13:39:17.000Z,1,-73.9476318359375,40.775150299072266,-73.9970703125,40.733951568603516,N,1062,35.4


In [0]:
%sql
select count(*) from learn_get_started_de_palvikrant235.taxi_etl_project.bronze_taxi_trips

count(*)
1458644


In [0]:
%sql
select count(*) from learn_get_started_de_palvikrant235.taxi_etl_project.silver_taxi_trips

count(*)
1458644


In [0]:
silver_table = "learn_get_started_de_palvikrant235.taxi_etl_project.silver_taxi_incremental"

print(silver_table)

learn_get_started_de_palvikrant235.taxi_etl_project.silver_taxi_incremental


In [0]:
bronze_table = "learn_get_started_de_palvikrant235.taxi_etl_project.bronze_taxi_incremental"

df_bronze = spark.table(bronze_table)

print("Bronze Records:", df_bronze.count())

Bronze Records: 1458654


In [0]:
from pyspark.sql import functions as F

In [0]:
print("Silver Total Records:",
      spark.table(silver_table).count())

print("BATCH_005 Records:",
      spark.table(silver_table)
      .filter(F.col("batch_id") == "BATCH_005")
      .count())

Silver Total Records: 1458654
BATCH_005 Records: 10
